<a href="https://colab.research.google.com/github/justamy20/scikit-learn-Cookbook/blob/main/Chapter_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 1: Konvensi Umum dan Elemen API scikit-learn

<div class="alert alert-info">
<b>Ringkasan Bab:</b> Bab ini memperkenalkan prinsip desain dasar dan elemen API utama dari scikit-learn. Kita akan mempelajari antarmuka konsisten yang menyatukan berbagai komponen seperti Estimator, Transformer, dan Pipeline.
</div>

## 1. Filosofi Desain scikit-learn
Pustaka ini dibangun di atas empat prinsip utama:
1.  **Consistency**: Semua algoritma mengikuti antarmuka yang sama (`fit`, `predict`, `transform`).
2.  **Simplicity**: Nilai default yang masuk akal memungkinkan prototyping cepat.
3.  **Modularity**: Komponen dapat disusun dan digabungkan secara bebas.
4.  **Reusability**: Komponen dapat digunakan kembali di proyek lain dengan mudah.

In [3]:
import numpy as np
import warnings
import sklearn

warnings.filterwarnings('ignore')
print(f"scikit-learn version: {sklearn.__version__}")
print(f"NumPy version: {np.__version__}")

scikit-learn version: 1.6.1
NumPy version: 2.0.2


---
## 2. Memahami Estimator
**Estimator** adalah objek apa pun dalam scikit-learn yang dapat belajar dari data menggunakan metode `fit()`.

### Supervised Learning: Linear Regression
Dalam supervised learning, estimator mempelajari pemetaan $y = f(X)$. Regresi linear mencari vektor bobot $w$ dan bias $b$ yang meminimalkan jumlah sisa kuadrat:

$$\min_{w, b} ||Xw + b - y||^2_2$$

In [4]:
from sklearn.linear_model import LinearRegression

# Data contoh (5 sampel, 1 fitur)
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

# Inisialisasi dan fitting model
model = LinearRegression()
model.fit(X, y)

# Prediksi untuk data baru
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)

print("Prediksi untuk X = [6, 7]:", predictions)

Prediksi untuk X = [6, 7]: [5.75 6.7 ]


### Unsupervised Learning: K-Means Clustering
Untuk unsupervised learning, kita sering menggunakan `fit_predict()`. Algoritma K-Means membagi $n$ observasi ke dalam $k$ klaster dengan meminimalkan *within-cluster sum of squares* (WCSS):

$$\sum_{i=1}^{n} \min_{\mu_j \in C} (||x_i - \mu_j||^2)$$

In [5]:
from sklearn.cluster import KMeans

# Menggunakan data X yang sama untuk clustering
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

print("Label klaster:", labels)
print("Pusat klaster:", kmeans.cluster_centers_.flatten())

Label klaster: [0 0 0 1 1]
Pusat klaster: [2.  4.5]


---
## 3. Transformer dan Metode `transform()`
**Transformer** adalah estimator yang mengubah data (preprocessing). Metode utamanya adalah:
* `fit(X)`: Mempelajari parameter.
* `transform(X)`: Menerapkan transformasi.
* `fit_transform(X)`: Melakukan keduanya sekaligus secara efisien.

**StandardScaler** melakukan normalisasi Z-score agar data terpusat pada nilai 0 dengan standar deviasi 1:
$$z = \frac{x - \mu}{\sigma}$$

In [6]:
from sklearn.preprocessing import StandardScaler

X_sample = np.array([[1, 2], [3, 4], [5, 6]])
scaler = StandardScaler()

# Melakukan fit dan transform sekaligus
X_scaled = scaler.fit_transform(X_sample)

print("Data asli:\n", X_sample)
print("\nData setelah di-scale:\n", X_scaled)
print("\nMean learned:", scaler.mean_)

Data asli:
 [[1 2]
 [3 4]
 [5 6]]

Data setelah di-scale:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Mean learned: [3. 4.]


---
## 4. Custom Transformers
Kita dapat membuat transformer sendiri dengan mewarisi `BaseEstimator` dan `TransformerMixin`. Hal ini memungkinkan integrasi yang mulus dengan ekosistem scikit-learn.


In [7]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class ValueClipper(BaseEstimator, TransformerMixin):
    """Memotong nilai fitur ke rentang [lower, upper]."""
    def __init__(self, lower=0.0, upper=1.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_is_fitted(self, 'is_fitted_')
        return np.clip(X, self.lower, self.upper)

# Contoh penggunaan
X_raw = np.array([[0.5, -0.3], [1.5, 0.7]])
clipper = ValueClipper(lower=0.0, upper=1.0)
print("Hasil Clipping:\n", clipper.fit_transform(X_raw))

Hasil Clipping:
 [[0.5 0. ]
 [1.  0.7]]


---
## 5. Pipeline dan Evaluasi Model
**Pipeline** menggabungkan beberapa langkah pemrosesan menjadi satu objek untuk mencegah kebocoran data (*data leakage*). Setelah fitting, kita bisa mengevaluasi model menggunakan metode `score()`.

Metrik $R^2$ untuk regresi dihitung sebagai:
$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$

<div class="alert alert-warning">
<b>Peringatan:</b> Jangan pernah memanggil fungsi fit() pada data testing saat menggunakan StandardScaler. Gunakan Pipeline agar proses ini ditangani secara otomatis dan aman!
</div>

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Membuat dataset sintetis
X_gen, y_gen = make_classification(n_samples=200, n_features=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_gen, y_gen, test_size=0.25)

# Membangun pipeline: Scaling -> Classification
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])

pipe.fit(X_train, y_train)
print(f"Akurasi Test Set dengan Pipeline: {pipe.score(X_test, y_test):.4f}")

# Inspeksi atribut model dari Linear Regression di awal (Bab 2)
print(f"\n--- Evaluasi Linear Regression Sebelumnya ---")
print(f"Coefficient (w): {model.coef_}")
print(f"Intercept (b):   {model.intercept_}")
print(f"R-squared score: {model.score(X, y):.4f}")

Akurasi Test Set dengan Pipeline: 0.8200

--- Evaluasi Linear Regression Sebelumnya ---
Coefficient (w): [0.95]
Intercept (b):   0.04999999999999938
R-squared score: 0.9810
